# Lab 2: Embeddings and document retrieval

Create embeddings for a small collection of notes, rank them against a question, and pass the retrieved text to the chat model.

**Prerequisites:** [setup](../docs/00_prework.md) and a working team key. Completing the full chat notebook is optional. Allow about 20 minutes.

The vectors in this exercise live in notebook memory. The [Object Storage lab](../docs/01_knowledge.md) and [PostgreSQL lab](../docs/02_memory.md) connect the same process to persistent services.

[Notebook index](README.md) · [API reference](../docs/inference.md) · [Troubleshooting](../docs/troubleshooting.md#inference)

These examples use the workshop service configuration. Confirm aliases and quotas with the organizers. Event details used as sample data are fictional.


## Step 1: Set up your environment

In [ ]:
# Install the packages used in this notebook.
%pip install -q "openai>=3,<4" python-dotenv numpy

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Run Jupyter from the workshop directory or its notebooks directory.
workshop_dir = Path.cwd().resolve()
if workshop_dir.name == "notebooks":
    workshop_dir = workshop_dir.parent
env_file = workshop_dir / ".env"
if not (workshop_dir / ".env.example").is_file() or not env_file.is_file():
    raise RuntimeError("Open Jupyter from the workshop directory after completing docs/00_prework.md.")
load_dotenv(env_file, override=True)

base_url = os.environ.get("OPENAI_BASE_URL", "").strip().rstrip("/")
api_key = os.environ.get("OPENAI_API_KEY", "").strip()
if not base_url or not api_key or api_key == "hk_paste_your_team_key_here":
    raise RuntimeError("Set OPENAI_BASE_URL and your team key in the workshop .env file.")
print("Endpoint:", base_url)
print("Team key loaded. Key value is hidden.")
client = OpenAI(base_url=base_url, api_key=api_key, timeout=60.0)


## Request reference

`POST /v1/embeddings` accepts `model="hackathon-embedding"` and an `input` string or list. The supplied deployment accepts up to 16 strings per request and returns 1,024-dimensional vectors. Read each vector from `data[i].embedding`.

In [ ]:
response = client.embeddings.create(
    model="hackathon-embedding",
    input=["first text", "second text"],
)
print(len(response.data), "vectors of", len(response.data[0].embedding), "numbers")

## Step 2: Embed one text

Inspect the response shape and vector length. The numbers represent the text for similarity comparisons; they are not a human-readable summary.

In [ ]:
import numpy as np

response = client.embeddings.create(
    model="hackathon-embedding",
    input="Akamai Cloud runs GPU instances in Seattle.",
)
vector = response.data[0].embedding

print("Model:      ", response.model)
print("Dimensions: ", len(vector))
print("First five: ", [round(v, 4) for v in vector[:5]])
print("Usage:      ", response.usage)

Embedding calls share the team's request quota. Check the [API reference](../docs/inference.md) for the deployment's separate request and token limits.

## Step 3: Batch texts

The supplied service accepts up to 16 inputs per call. The following intentionally oversized request demonstrates its validation error.

In [ ]:
import openai

try:
    client.embeddings.create(model="hackathon-embedding", input=[f"sentence {i}" for i in range(17)])
except openai.APIStatusError as e:
    print("Status:", e.status_code)
    print("Body:  ", e.response.text[:200])

The batching helper splits the list into groups of 16 and returns a NumPy array in input order. It also handles `429` with bounded retries.

In [ ]:
import random
import time

embedding_client = client.with_options(max_retries=0)

BATCH = 16  # Server-side maximum inputs per request.

def embed(texts, attempts=5):
    """Embed any number of strings, 16 per request, preserving order. Retries on 429."""
    vectors = []
    for start in range(0, len(texts), BATCH):
        batch = texts[start:start + BATCH]
        for attempt in range(attempts):
            try:
                r = embedding_client.embeddings.create(model="hackathon-embedding", input=batch)
                break
            except openai.RateLimitError as e:
                if attempt == attempts - 1:
                    raise
                # This service returns delay values in seconds.
                wait = float(e.response.headers.get("retry-after") or e.response.headers.get("x-ratelimit-reset") or 3) + random.uniform(0, 1)
                print(f"429 on batch starting at {start}; retrying in {wait:.0f}s")
                time.sleep(wait)
        else:
            raise RuntimeError("Embedding failed after repeated 429 responses")
        vectors.extend(item.embedding for item in r.data)
    return np.array(vectors, dtype=np.float32)

sample = embed([f"sentence {i}" for i in range(20)])
print("Shape:", sample.shape)

## Step 4: Search by meaning

Normalize document and query vectors, then compare them with a dot product. This is cosine similarity: 1 means the vectors point in the same direction, and 0 means they are orthogonal. Similarity scores are not calibrated probabilities of relevance.

The corpus below contains fictional event notes. Embed them once, then rank them for each query.

In [ ]:
# Fictional event notes for this exercise.
corpus = [
    "The hackathon starts Saturday at 9 AM in the main hall.",
    "Teams may have up to five members.",
    "Submissions are due Sunday at noon through the Devpost page.",
    "Breakfast and lunch are provided both days; dinner is on your own.",
    "Each team receives one API key for the inference endpoint.",
    "The chat model accepts up to about 30,000 tokens of input.",
    "Image generation is limited to six requests per minute per team.",
    "Judging criteria are technical difficulty, design, and usefulness.",
    "Parking is free in Lot C on the north side of the building for attendees with a badge.",
    "The Wi-Fi network is HackMidwest and the password is printed on your badge.",
    "Mentors wear green lanyards and rotate between tables.",
    "Prizes are announced at the closing ceremony on Sunday at 4 PM.",
]

doc_vectors = embed(corpus)
doc_vectors /= np.linalg.norm(doc_vectors, axis=1, keepdims=True)  # Unit length, so dot product = cosine.

def search(query, k=3):
    q = embed([query])[0]
    q /= np.linalg.norm(q)
    scores = doc_vectors @ q
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), corpus[i]) for i in top]

for score, text in search("When do we have to hand in our project?"):
    print(f"{score:.3f}  {text}")

Inspect the highest-ranked note. Semantic similarity can match different wording, such as “hand in our project” and “submissions are due.” Check relevance yourself; the first result is not automatically correct.

In [ ]:
for score, text in search("Is there food?"):
    print(f"{score:.3f}  {text}")

## Step 5: Answer from retrieved notes

First ask without providing the notes. The model does not have the fictional event details. Compare that response with the retrieval-based response below.

In [ ]:
direct = client.chat.completions.create(
    model="hackathon-chat",
    messages=[{"role": "user", "content": "Where can I park at HackMidwest? One sentence."}],
    max_tokens=60,
)
print(direct.choices[0].message.content)

Retrieve relevant notes, include them in the prompt, and instruct the model to use those notes. This pattern is called retrieval-augmented generation, or RAG. Retrieval supplies context; it does not guarantee a correct answer.

In [ ]:
def answer(question, k=3):
    hits = search(question, k)
    context = "\n".join(f"- {text}" for _, text in hits)
    prompt = (
        "Answer the question using only the notes below. "
        "If the notes do not contain the answer, say you do not know.\n\n"
        f"Notes:\n{context}\n\nQuestion: {question}"
    )
    r = client.chat.completions.create(
        model="hackathon-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=120,
    )
    return r.choices[0].message.content.strip(), hits

reply, hits = answer("Where can I park at HackMidwest?")
print(reply)
print("\nRetrieved notes:")
for score, text in hits:
    print(f"  {score:.3f}  {text}")

In [ ]:
reply, hits = answer("What is the prize money?")
print(reply)

The prize-money question is not answered by the notes. Check whether the model acknowledges the missing information. Include questions with missing or conflicting evidence when evaluating your application.

## Use this in an application

- Chunk long text before embedding it. The supplied deployment documents 16 inputs and 8,192 total tokens per request; do not rely on silent truncation.
- Use the same embedding model for documents and queries. Re-embed stored text when changing the model.
- Store original documents in [Object Storage](../docs/01_knowledge.md).
- Store searchable chunks and vectors in [PostgreSQL with pgvector](../docs/02_memory.md) when they need to survive restarts.
- Check retrieved sources and unsupported questions as well as ordinary answers.

## Validation

In [ ]:
score, text = search("How many people can be on a team?", k=1)[0]
assert len(vector) == 1024
assert "five members" in text, text
print(f"Top match ({score:.3f}): {text}")
print("Lab 2 complete.")

## Clean up and continue

No cloud resources were created. Restarting the kernel removes the in-memory vectors.

Continue with the [Object Storage lab](../docs/01_knowledge.md), then optionally [persist the vectors](../docs/02_memory.md). The starter implements this flow in `agent/knowledge.py` and `agent/memory.py`.